# Procesamiento de Vídeo con OpenCV

## Leer Archivos de Vídeo

`cv2.VideoCapture` funciona con **archivos**, **webcams** (pasa un índice entero — `0` para la primera cámara), y **flujos de red** (URLs RTSP/HTTP). La API es idéntica en los tres casos: abrir, leer fotogramas en un bucle, liberar cuando termines.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os

In [ ]:
# Descargar un vídeo de muestra
video_url = "https://www.w3schools.com/html/mov_bbb.mp4"
video_path = "../artifacts/outputs/sample_video.mp4"
os.makedirs(os.path.dirname(video_path), exist_ok=True)
if not os.path.exists(video_path):
    urllib.request.urlretrieve(video_url, video_path)
    print(f"Descargado: {video_path}")

In [ ]:
# Abrir archivo de vídeo
cap = cv2.VideoCapture(video_path)
# Obtener propiedades del vídeo usando constantes CAP_PROP_*.
# Otras constantes útiles: CAP_PROP_POS_FRAMES (saltar a un fotograma),
# CAP_PROP_BRIGHTNESS / CONTRAST / SATURATION (solo controles de cámara).
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Resolución: {width}x{height}")
print(f"FPS: {fps}")
print(f"Fotogramas totales: {total_frames}")
print(f"Duración: {total_frames/fps:.2f} segundos")
# Libera el manejador de archivo para que el SO pueda liberar el recurso.
# Llama siempre a esto — incluso si solo lees unos pocos fotogramas.
cap.release()

## Procesamiento Fotograma a Fotograma

In [ ]:
cap = cv2.VideoCapture(video_path)
# Leer los primeros 5 fotogramas.
# cap.read() devuelve una tupla (ret, frame):
#   ret   — bool; False cuando el vídeo termina o la fuente no está disponible.
#   frame — la imagen decodificada en formato BGR, o None cuando ret es False.
frames = []
for i in range(5):
    ret, frame = cap.read()
    if ret:
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
# Libera el archivo de vídeo para que el SO pueda liberar el recurso.
cap.release()

# Mostrar fotogramas
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f'Fotograma {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Aplicar Filtros al Vídeo

In [ ]:
def process_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Canny usa dos umbrales de histéresis:
    #   50  — límite inferior: los bordes por debajo de este siempre se descartan.
    #   150 — límite superior: los bordes por encima de este siempre se conservan.
    # Los píxeles en el medio se mantienen solo si se conectan a un borde fuerte (>150).
    # Regla empírica: umbral superior ≈ 2–3× inferior.
    edges = cv2.Canny(gray, 50, 150)
    return edges

cap = cv2.VideoCapture(video_path)
# Procesar fotogramas
processed_frames = []
for i in range(5):
    ret, frame = cap.read()
    if ret:
        processed = process_frame(frame)
        processed_frames.append(processed)
cap.release()

# Mostrar
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, (ax, frame) in enumerate(zip(axes, processed_frames)):
    ax.imshow(frame, cmap='gray')
    ax.set_title(f'Bordes {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Escribir Vídeo de Salida

`VideoWriter` codifica fotogramas en un archivo de vídeo comprimido. El parámetro clave es el **codec fourcc** — un código de 4 caracteres que identifica el algoritmo de compresión:

| Código | Algoritmo | Contenedor |
|------|-----------|-----------| 
| `mp4v` | MPEG-4 Parte 2 | `.mp4` |
| `avc1` / `H264` | H.264 | `.mp4` |
| `XVID` | Xvid/MPEG-4 | `.avi` |
| `MJPG` | Motion JPEG | `.avi` |

La disponibilidad del codec depende de lo que esté instalado en tu sistema.

In [ ]:
cap = cv2.VideoCapture(video_path)
output_path = '../artifacts/outputs/output_edges.mp4'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
# VideoWriter_fourcc(*'mp4v') es abreviatura de VideoWriter_fourcc('m','p','4','v').
# El * desempaqueta la cadena en cuatro argumentos char separados.
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# VideoWriter(archivo, fourcc, fps, tamañoFrame, esColor)
# isColor=False: los fotogramas son de un solo canal (escala de grises). Debe coincidir con lo que escribes.
# frameSize (ancho, alto) debe coincidir exactamente — una discrepancia corrompe silenciosamente la salida.
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=False)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    edges = process_frame(frame)
    out.write(edges)
# cap.release() libera el manejador del archivo de entrada.
# out.release() vacía los búferes del codificador y escribe los metadatos finales del contenedor
# (el átomo MOOV en MP4) — omitir esto produce un archivo irreproducible.
cap.release()
out.release()
print(f"Vídeo guardado en: {output_path}")
# Limpieza
# for f in [video_path, output_path]:
#     if os.path.exists(f):
#         os.remove(f)

## Captura por Webcam

> **Nota:** Esta celda abre una ventana GUI y bloquea la ejecución hasta que pulses `q`. Requiere una **pantalla** y una **cámara física** — no funcionará en entornos sin cabeza o notebooks en la nube.

In [ ]:
# ⚠️ Requiere pantalla con GUI y webcam física.
# Ejecuta esto en un entorno local — no en un servidor sin cabeza o notebook en la nube.
# Índice 0 = primera cámara detectada. Usa 1, 2 ... para cámaras adicionales.
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow('Webcam', frame)
    # waitKey(1): espera 1 ms para una pulsación de tecla — también necesario para actualizar la ventana GUI.
    # & 0xFF enmascara a 8 bits por compatibilidad entre plataformas.
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
# Libera la cámara para que otras aplicaciones puedan usarla.
cap.release()
# Cierra todas las ventanas creadas por cv2.imshow().
cv2.destroyAllWindows()

## Sustracción de Fondo

La sustracción de fondo identifica qué píxeles en un fotograma están en movimiento (primer plano) comparando cada fotograma con un modelo aprendido de la escena estática (fondo).

Intuición de parámetros:
- `history` — el número de fotogramas pasados utilizados para actualizar el modelo de fondo. Historia corta (50–100 fotogramas): se adapta rápidamente pero puede absorber objetos de movimiento lento como fondo. Historia larga (500+): modelo de fondo estable para cámaras estáticas.
- `varThreshold` — la distancia de Mahalanobis más allá de la cual un píxel se clasifica como primer plano. Valor bajo: sensible, detecta movimientos sutiles pero también ruido. Valor alto: solo las grandes desviaciones se marcan como primer plano.
- `detectShadows=True`: las sombras se etiquetan como una clase separada (píxeles grises, valor 127) en lugar de mezclarse con el primer plano. Útil si quieres ignorar las sombras en el posprocesamiento.

In [ ]:
cap = cv2.VideoCapture(video_path)
# MOG2 = Mixture of Gaussians v2.
#   history      — número de fotogramas pasados para construir el modelo de fondo.
#                  Mayor = adaptación más lenta (bueno para cámaras estáticas; malo cuando la iluminación cambia).
#   varThreshold — umbral de distancia de Mahalanobis. Mayor = menos sensible (menos falsas alarmas).
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=50)

masks = []
for i in range(10):
    ret, frame = cap.read()
    if not ret:
        break
    # apply() actualiza el modelo Y devuelve la máscara de primer plano.
    # Valores de la máscara: 255 = primer plano en movimiento, 0 = fondo, 127 = sombra detectada.
    fg_mask = bg_subtractor.apply(frame)
    masks.append(fg_mask)
cap.release()

# Mostrar las últimas máscaras (después de que el modelo aprende el fondo)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, ax in enumerate(axes):
    ax.imshow(masks[6+i], cmap='gray')
    ax.set_title(f'Máscara de movimiento {6+i}')
    ax.axis('off')
plt.suptitle('Sustracción de Fondo')
plt.tight_layout()
plt.show()

## Resumen

| Tarea | Función |
|------|----------|
| Leer vídeo | `cv2.VideoCapture(path)` |
| Escribir vídeo | `cv2.VideoWriter(path, fourcc, fps, size)` |
| Leer fotograma | `cap.read()` |
| Propiedades del vídeo | `cap.get(cv2.CAP_PROP_*)` |
| Sustracción de fondo | `cv2.createBackgroundSubtractorMOG2()` |